# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q beautifulsoup4 requests
    print('Packages installed.')


# Web Scraping for Social Science: An In-Depth Guide

Web scraping is a powerful tool for collecting digital trace data. This notebook moves beyond simple tutorials to cover robust, ethical, and structured data collection.

## Key Concepts Covered
1.  **Ethical Scraping**: `robots.txt` and rate limiting.
2.  **HTML Parsing**: Navigating the DOM with `BeautifulSoup`.
3.  **Pagination**: looping through multiple pages.
4.  **Error Handling**: Managing timeouts and missing elements.
5.  **Data Structuring**: Saving results to a clean CSV.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# Base URL for a site designed for scraping practice
BASE_URL = 'http://quotes.toscrape.com'

## 1. Ethical Considerations

Before scraping, always checks the website's `robots.txt` file (e.g., `website.com/robots.txt`) to see allowed paths. 

**Best Practices:**
-   **Identify yourself**: Use a User-Agent string with your contact info.
-   **Respect rate limits**: Add sleep timers between requests to avoid crashing the server.
-   **Public data only**: Do not scrape private user data behind logins without explicit consent.

In [ ]:
headers = {
    'User-Agent': 'SocialScienceResearchBot/1.0 (researcher@university.edu)'
}

## 2. Scraping a Single Page
We start by fetching one page and extracting the desired elements.

In [ ]:
def get_soup(url):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status() # Raise error for 404/500 codes
        return BeautifulSoup(response.text, 'html.parser')
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

soup = get_soup(BASE_URL)
if soup:
    print("Page title:", soup.title.text.strip())

## 3. Parsing Logic
We inspect the HTML structure (Right-click -> Inspect on the web page) to find the tags holding our data.

In [ ]:
def parse_quotes(soup):
    quotes_data = []
    quote_blocks = soup.find_all('div', class_='quote')
    
    for block in quote_blocks:
        text = block.find('span', class_='text').get_text(strip=True)
        author = block.find('small', class_='author').get_text(strip=True)
        
        # Extract tags list
        tags = [tag.get_text(strip=True) for tag in block.find_all('a', class_='tag')]
        
        quotes_data.append({
            'Quote': text,
            'Author': author,
            'Tags': ", ".join(tags)
        })
    return quotes_data

if soup:
    sample_data = parse_quotes(soup)
    print(f"Found {len(sample_data)} quotes on the first page.")
    print(sample_data[0])

## 4. Handling Pagination
To build a dataset, we need to iterate through multiple pages. We loop until there is no "Next" button or we reach a limit.

In [ ]:
all_quotes = []
page_num = 1
max_pages = 5 # Limit for this demo

while page_num <= max_pages:
    print(f"Scraping page {page_num}...")
    url = f"{BASE_URL}/page/{page_num}/"
    soup = get_soup(url)
    
    if not soup:
        break
        
    quotes_on_page = parse_quotes(soup)
    if not quotes_on_page:
        print("No more quotes found.")
        break
        
    all_quotes.extend(quotes_on_page)
    
    # Check for 'Next' button (robustness check)
    next_btn = soup.find('li', class_='next')
    if not next_btn:
        print("Reached last page.")
        break
        
    # POLITE SCRAPING: Sleep to respect server
    time.sleep(random.uniform(1, 3))
    
    page_num += 1

print(f"\nScraping complete. Collected {len(all_quotes)} quotes.")

## 5. Saving Data
Convert to DataFrame and save to CSV for analysis.

In [ ]:
df = pd.DataFrame(all_quotes)
df.to_csv('scraped_quotes.csv', index=False)
print(df.head())

## 6. Mini-Analysis
Let's see who the most quoted authors are.

In [ ]:
top_authors = df['Author'].value_counts().head(5)
print("Top 5 Authors:")
print(top_authors)